# Project Structure

## Table of Contents

1. [Why Structure Matters](#1)
2. [The Recommended Layout](#2)
3. [The src Layout: Why Bother?](#3)
4. [pyproject.toml: One Config File to Rule Them All](#4)
5. [__init__.py: What Goes Where](#5)
6. [Exercise: Restructure the Warm-Up Code](#6)

---
## 1. Why Structure Matters <a id="1"></a>

A notebook is a scratch pad. It's fine for exploration.

But once you need to:
- Re-use code from another notebook
- Test a function in isolation
- Schedule a job to run overnight
- Let a colleague run your pipeline on their machine
- Deploy a model to an API

...you need a *project*, not a notebook collection.

The single rule: **if code appears in more than one place, it should be in a module**.

---
## 2. The Recommended Layout <a id="2"></a>

Adapted from [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/), with modern tooling updates:

```
my_project/
├── pyproject.toml          <- Package metadata + dependencies (replaces
│                              setup.py + requirements.txt)
├── README.md
├── .gitignore
│
├── data/
│   ├── raw/                <- Original data — NEVER modify these files
│   ├── external            <- Same as raw, but for external data
│   ├── interim/            <- Intermediate cleaned/filtered data
│   └── processed/          <- Final features ready for modelling
│
├── docs/                   <- Automatic documentation
│
├── models/                 <- Serialised trained models (.pkl, .joblib)
│
├── notebooks/              <- Exploration and reporting
│   └── m01_mtr_eda.ipynb        <- Naming: number + author initials + description
│
├── reports/
│   └── figures/            <- Generated charts
│
└── src/
    └── my_project/         <- The importable Python package
        ├── __init__.py
        ├── config.py        <- Constants, paths, env variables
        ├── data.py          <- Data loading / Dataset class
        ├── features.py      <- Feature engineering
        ├── models/
        │   ├── __init__.py
        │   ├── train.py
        │   └── predict.py
        └── plots.py
```

### Key decisions

- **`data/raw` is read-only**. Ever. Document where it came from and never commit it to git.
- **Notebooks are for exploration**. Reusable code lives in `src/`. The notebook imports from `src/`, it doesn't duplicate it.
- **`models/` contains artefacts**, not code. Don't commit large `.pkl` files to git either — use DVC (see notebook 13).

---
## 3. The src Layout: Why Bother? <a id="3"></a>

You might wonder: why not just put all the Python files in the root, like this?

```
my_project/
├── data.py         # <- flat layout
├── features.py
├── train.py
└── notebooks/
```

The problem: Python adds the current directory to `sys.path`. This means `import data` in a notebook might import *your* `data.py` **or** a totally different `data` package depending on where you run the notebook from. This creates a class of "works on my machine" bugs that are hard to trace.

The `src` layout forces you to *install* your package (even in development mode), which means Python always imports from the right place.

In [ ]:
# How to install your own package in development mode:
# (Run this in the terminal, in the project root)
#
# uv sync        (installs everything from uv.lock, including the src/ package)
#
# uv uses the pyproject.toml to know where the package lives (src/churn/).
# After sync, any notebook can do:
#
#   from my_project.data import Dataset
#   from my_project.features import StandardScaler
#
# ...and it will always import from src/my_project/, not from wherever you happen to be.

print("uv sync makes src/churn importable from anywhere in the project")

---
## 4. pyproject.toml: One Config File to Rule Them All <a id="4"></a>

`pyproject.toml` replaces `setup.py`, `setup.cfg`, and `requirements.txt`. Modern Python packaging uses it for everything: dependencies, tool configuration, build settings.

Here's the `pyproject.toml` from this repo as a reference:

In [ ]:
import tomllib
from pathlib import Path
import subprocess

# Read the actual pyproject.toml from this repo
toml_path = Path(".").resolve().parent / "pyproject.toml"
if toml_path.exists():
    with open(toml_path, "rb") as f:
        config = tomllib.load(f)
    import json
    print(json.dumps(config, indent=2, default=str))
else:
    print("pyproject.toml not found at expected path — showing template instead")

A typical `pyproject.toml` for a DS project:

```toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my_churn_project"
version = "0.1.0"
python_requires = ">=3.11"
dependencies = [
    "pandas>=2.0",
    "scikit-learn>=1.4",
    "pydantic>=2.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=8",
    "pytest-cov",
    "ruff",      # linter + formatter
    "mypy",      # type checker
]

[tool.setuptools.packages.find]
where = ["src"]     # tells setuptools where to find the package

[tool.ruff]
line-length = 100

[tool.mypy]
strict = false
ignore_missing_imports = true
```

**Key difference from `requirements.txt`**: `pyproject.toml` specifies *minimum* versions (constraints). `requirements.txt` pins *exact* versions (for reproducible deploys). You can generate a lockfile from `pyproject.toml` using `pip-compile` or `uv`.

---
## 5. __init__.py: What Goes Where <a id="5"></a>

`__init__.py` makes a directory a Python package. But *what goes in it* matters.

In [ ]:
# Good __init__.py patterns:

# 1. Re-export the public API
# src/my_project/__init__.py
#
# from .data import Dataset
# from .features import StandardScaler, MinMaxScaler
# from .config import PROCESSED_DATA_PATH
#
# __all__ = ["Dataset", "StandardScaler", "MinMaxScaler", "PROCESSED_DATA_PATH"]

# This lets users do:
# from my_project import Dataset
# instead of:
# from my_project.data import Dataset

# 2. Keep it short. Never put logic in __init__.py.
# Never do expensive imports (like loading a model from disk).

# 3. Version tagging
# __version__ = "0.1.0"

print("__init__.py: short, re-exports only, no logic")

---
## 6. Exercise: Restructure the Warm-Up Code <a id="6"></a>

Look at the warm-up notebook (00_Warmup_Whats_Wrong.ipynb). Extract the reusable parts into a proper module structure.

**Tasks:**
1. Create a directory structure in `src/churn_project/`
2. Create `data.py` with a `load_churn_data(path)` function
3. Create `features.py` with an `encode_categoricals(df, columns)` function
4. Create `models.py` with a `train_random_forest(X, y, n_estimators=100)` function that returns a trained model
5. Create `__init__.py` that re-exports all three functions

**Sketch the directory tree below before writing any code:**

```
churn_project/
├── pyproject.toml
├── src/
│   └── churn_project/
│       ├── __init__.py
│       └── ...
└── notebooks/
    └── 00_Warmup.ipynb
```

In [ ]:
# Create the module structure using Python
import os
from pathlib import Path

base = Path("/tmp/churn_project_exercise")

# Create directories
(base / "src" / "churn_project").mkdir(parents=True, exist_ok=True)
(base / "notebooks").mkdir(exist_ok=True)
(base / "data" / "raw").mkdir(parents=True, exist_ok=True)

# Write data.py
(base / "src" / "churn_project" / "data.py").write_text('''import pandas as pd

def load_churn_data(path: str) -> pd.DataFrame:
    """Load raw churn CSV and return a DataFrame with basic validation."""
    df = pd.read_csv(path)
    required_cols = {"customerID", "Churn"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")
    return df
''')

# Write features.py
(base / "src" / "churn_project" / "features.py").write_text('''import pandas as pd
from sklearn.preprocessing import LabelEncoder

def encode_categoricals(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Label-encode a list of categorical columns. Returns a copy."""
    df = df.copy()
    le = LabelEncoder()
    for col in columns:
        df[col] = le.fit_transform(df[col].astype(str))
    return df
''')

# Write models.py
(base / "src" / "churn_project" / "models.py").write_text('''import pandas as pd
from sklearn.ensemble import RandomForestClassifier

def train_random_forest(
    X: pd.DataFrame,
    y: pd.Series,
    n_estimators: int = 100,
    random_state: int = 42,
) -> RandomForestClassifier:
    """Train and return a fitted RandomForestClassifier."""
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
    clf.fit(X, y)
    return clf
''')

# Write __init__.py
(base / "src" / "churn_project" / "__init__.py").write_text('''from .data import load_churn_data
from .features import encode_categoricals
from .models import train_random_forest

__version__ = "0.1.0"
__all__ = ["load_churn_data", "encode_categoricals", "train_random_forest"]
''')

print("Structure created:")
for p in sorted(base.rglob("*.py")):
    print(" ", p.relative_to(base))